In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS catalogmeteo.gold;

In [0]:
%sql
select * from catalogmeteo.silver.weather_observations_24 where station_id <> 'RM001' and station_id <> 'MI001'

### CALCULATION OF AVERAGE TEMPERATURE, AVERAGE WIND AND AVERAGE PRESSURE PER AIRPORT

In [0]:
from pyspark.sql import functions as f

# 1. Load DataFrames
obs_df = spark.table("catalogmeteo.silver.weather_observations_24") \
    .withColumn("period", f.date_format(f.col("date"), "yyyy-MM"))

meta_df = spark.table("catalogmeteo.bronze.weather_stations_metadata")

# 2. Join using specific DataFrame references to avoid ambiguity
# We join on station_id, which Spark handles automatically if it's the only common column
joined_df = meta_df.join(obs_df, "station_id", "inner")

# 3. Pivot with explicit column references
weather_pivot = joined_df.groupBy(
    meta_df.station_id,    # Explicitly use meta_df version
    meta_df.station_name, 
    meta_df.city,          # This fixes the [AMBIGUOUS_REFERENCE] error
    meta_df.elevation_m
).pivot("period").agg(
    f.round(f.avg("temperature_c"), 2).alias("avg_temp"),
    f.round(f.avg("humidity_pct"), 2).alias("avg_humidity"),
    f.round(f.avg("wind_speed_kmh"), 2).alias("avg_wind_speed")
)

display(weather_pivot)

In [0]:
%sql
select meta.station_id, meta.station_name, meta.city, meta.elevation_m, 
avg(obs.temperature_c) as avg_temp, 
avg(obs.humidity_pct) as avg_humidity, 
avg(obs.wind_speed_kmh) as avg_wind_speed
from catalogmeteo.bronze.weather_stations_metadata as meta
left join catalogmeteo.silver.weather_observations_24 as obs
on meta.station_id = obs.station_id
group by meta.station_id, meta.station_name, meta.city, meta.elevation_m
    
